# Locy Neural Use Case: Drug-Drug Interaction Risk Scoring (Rust)

Compact Rust counterpart to the Python DDI flagship. Same shape: inline pairwise-interaction graph + a `MockClassifier` + `CREATE MODEL` + `CALIBRATE` + `VALIDATE` + `EXPLAIN`. In production this is exactly where you'd plug in a small MLP head over precomputed R-GCN drug embeddings.


## 1) Setup + Schema


In [ ]:
use std::sync::Arc;
use uni_db::{DataType, Uni, Result};
use uni_locy::{LocyConfig, MockClassifier, NeuralClassifier, FeatureValue};

let db = Uni::in_memory().build().await?;
db.schema()
    .label("InteractionRecord")
        .property("pair_id", DataType::String)
        .property("severity", DataType::Float64)
        .property("is_dangerous", DataType::Bool)
    .done()
    .apply()
    .await?;


## 2) Seed Interaction Records


In [ ]:
let session = db.session();
let tx = session.tx().await?;
let rows: &[(&str, f64, bool)] = &[
    ("P01", 0.55, true),  ("P02", 0.18, false),
    ("P03", 0.60, true),  ("P04", 0.22, false),
    ("P05", 0.42, true),  ("P06", 0.25, false),
    ("P07", 0.65, true),  ("P08", 0.20, false),
    ("P09", 0.50, true),  ("P10", 0.30, false),
];
for (pid, sev, danger) in rows {
    let q = format!(
        "CREATE (:InteractionRecord {{pair_id: '{}', severity: {}, is_dangerous: {}}})",
        pid, sev, danger
    );
    tx.execute(&q).await?;
}
tx.commit().await?;


## 3) Register the Interaction Classifier


In [ ]:
let classifier: Arc<dyn NeuralClassifier> = Arc::new(
    MockClassifier::new("interaction_score", |inp| {
        let sev = match inp.features.get("rec") {
            Some(FeatureValue::Float(v)) => *v,
            _ => 0.0,
        };
        let z = (sev - 0.40) * 8.0 - 0.3;
        let p = 1.0 / (1.0 + (-z).exp());
        let p_sharp = 1.0 / (1.0 + (-4.0 * (p - 0.5)).exp());
        p_sharp.clamp(0.0, 1.0)
    })
);
let mut config = LocyConfig::default();
config.classifier_registry.insert("interaction_score".to_string(), classifier);


## 4) CREATE MODEL + Score


In [ ]:
let program = r#"
CREATE MODEL interaction_score AS
  INPUT (rec)
  FEATURES rec.severity
  OUTPUT PROB risk
  USING xervo('classify/ddi-v1')

CREATE RULE scored_interactions AS
  MATCH (rec:InteractionRecord)
  YIELD KEY rec, interaction_score(rec.severity) AS risk
"#;
let result = session.locy_with(program).with_config(config.clone()).run().await?;
let n = result.derived().get("scored_interactions").map(|v| v.len()).unwrap_or(0);
println!("Scored {} interaction pairs", n);


## 5) CALIBRATE


In [ ]:
let calibrate_program = r#"
CREATE MODEL interaction_score AS
  INPUT (rec)
  FEATURES rec.severity
  OUTPUT PROB risk
  USING xervo('classify/ddi-v1')

CALIBRATE interaction_score
  ON MATCH (rec:InteractionRecord)
  TARGET rec.is_dangerous
  METHOD platt_scaling
"#;
let calib = session.locy_with(calibrate_program).with_config(config.clone()).run().await?;
println!("command_results: {} entries", calib.command_results().len());


## 6) EXPLAIN One Dangerous Interaction


In [ ]:
let explain_program = format!("{}{}", program, r#"

EXPLAIN RULE scored_interactions WHERE rec.pair_id = 'P01'
"#);
let explain = session.locy_with(&explain_program).with_config(config).run().await?;
println!("EXPLAIN command_results: {}", explain.command_results().len());
